In [1]:
from typing import TypedDict, cast

import pandas as pd


class GroundTruthRecord(TypedDict):
    question: str
    document: str


faq_ground_truth_df = pd.read_csv("data/faq_ground_truth.csv")

display(faq_ground_truth_df.head())

faq_ground_truth: list[GroundTruthRecord] = cast(
    list[GroundTruthRecord],
    faq_ground_truth_df.to_dict(orient="records"))

,question,document
0,I just found this course late — can I still st...,74eb249bbf
1,Is it too late to enroll if I discovered the c...,74eb249bbf
2,Can I still take part in the course even thoug...,74eb249bbf
3,"If I join the course late, am I still eligible...",74eb249bbf
4,What’s the deadline for submitting the project...,74eb249bbf


In [2]:
from typing import cast

from pydash import filter_

from lib.sources import load_faq_data
from lib.types import JSONDocument


predicate = {"course": "llm-zoomcamp"}

documents = filter_(load_faq_data(), predicate)

In [ ]:
from lib.search import LexicalSearchConfig, MinsearchLexicalSearchTool


minsearch_lexical_search_tool = MinsearchLexicalSearchTool.from_documents(
    cast(list[JSONDocument], documents),
    text_fields=["question", "answer"],
    keyword_fields=["course"],
    config=LexicalSearchConfig(
        num_results=6,
        boost_dict={
            "question": 1.0,
            "answer": 3.0,
        }
    )
)

In [4]:
question = faq_ground_truth[0]["question"]

print(question)

print(minsearch_lexical_search_tool.search(question))

I just found this course late — can I still start it and join in now?
[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '04919992b3', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How should I start the course and follow the weekly workflow?', 'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-z

In [5]:
from lib.sources import FAQDocument


def are_results_relevant(
        results: list[FAQDocument],
        ground_truth: GroundTruthRecord) -> list[int]:
    return [int(result["id"] == ground_truth["document"]) for result in results]

In [6]:
results = cast(
    list[FAQDocument],
    minsearch_lexical_search_tool.search(faq_ground_truth[0]["question"]))

print(results)

print(faq_ground_truth[0])

are_results_relevant(results, faq_ground_truth[0])

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '04919992b3', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How should I start the course and follow the weekly workflow?', 'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson video

[1, 0, 0, 0, 0, 0]

In [26]:
from lib.search import SearchTool


def compute_total_relevance(
        faq_ground_truth: list[GroundTruthRecord],
        search_tool: SearchTool
        ) -> list[list[int]]:
    total_relevance_matrix: list[list[int]] = []
    for ground_truth_record in faq_ground_truth:
        results = cast(
            list[FAQDocument],
            search_tool.search(ground_truth_record["question"]))
        total_relevance_matrix.append(
            are_results_relevant(results, ground_truth_record))
    return total_relevance_matrix


relevances_with_lexical_search = compute_total_relevance(
    faq_ground_truth, minsearch_lexical_search_tool)

In [8]:
from scripts.embedder import Embedder

# encoder = SentenceTransformer("all-MiniLM-L6-v2")
encoder = Embedder("models/Xenova/all-MiniLM-L6-v2")

2026-06-28 21:30:33.378102934 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [ ]:
from lib.search import MinsearchSemanticSearchTool, SemanticSearchConfig


minsearch_semantic_search_tool = MinsearchSemanticSearchTool.from_documents(
    cast(list[JSONDocument], documents),
    encoder=encoder,
    text_fields=["question", "answer"],
    keyword_fields=["course"],
    config=SemanticSearchConfig(
        num_results=6,
    )
)

  0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
relevances_with_semantic_search = compute_total_relevance(
    faq_ground_truth, minsearch_semantic_search_tool)

  0%|          | 0/425 [00:00<?, ?it/s]

In [11]:
def compute_hit_rate(relevances: list[list[int]]) -> float:
    relevance_per_question = [any(relevance) for relevance in relevances]
    hit_rate = sum(relevance_per_question) / len(relevances)
    return hit_rate

def compute_mrr(relevances: list[list[int]]) -> float:
    total_score = 0.0
    for relevance in relevances:
        for rank in range(len(relevance)):
            if relevance[rank] == 1:
                score = 1 / (rank + 1)
                total_score += score
                break
    return total_score / len(relevances)

In [12]:
print(compute_hit_rate(relevances_with_lexical_search))
print(compute_hit_rate(relevances_with_semantic_search))
print(compute_mrr(relevances_with_lexical_search))
print(compute_mrr(relevances_with_semantic_search))

0.8847058823529412
0.9858823529411764
0.7545098039215685
0.9231372549019607


In [13]:
from pydash import find

print(relevances_with_semantic_search[-1])

print(faq_ground_truth[-1])

predicate = {"id": faq_ground_truth[-1]["document"]}

print(find(documents, predicate))

results = cast(
    list[FAQDocument],
    minsearch_semantic_search_tool.search(faq_ground_truth[-1]["question"])
)

print(are_results_relevant(results, faq_ground_truth[-1]))

[1, 0, 0, 0, 0, 0]
{'question': 'Is there a workaround for the requests version issue when using lancedb in this dlt session?', 'document': '4b30b918bc'}
{'id': '4b30b918bc', 'course': 'llm-zoomcamp', 'section': 'Workshop: Open-Source Data Ingestion (dlt)', 'question': 'Error: How to fix requests library only installs v2.28 instead of v2.32 required for lancedb?', 'answer': 'If you encounter a 401 Client Error, it may indicate the need to grant access to the key or that the key is incorrect.\n\nTo install the correct version directly from the source, use the following command:\n\n```bash\npip install "requests @ https://github.com/psf/requests/archive/refs/tags/v2.32.3.zip"\n```'}
[1, 0, 0, 0, 0, 0]


In [14]:
def evaluate(
        faq_ground_truth: list[GroundTruthRecord],
        search_function: SearchTool):
    
    total_relevance = compute_total_relevance(faq_ground_truth, search_function)

    return {
        "hit_rate": compute_hit_rate(total_relevance),
        "mrr": compute_mrr(total_relevance),
    }

In [15]:
print(evaluate(faq_ground_truth, minsearch_lexical_search_tool))

print(evaluate(faq_ground_truth, minsearch_semantic_search_tool))

  0%|          | 0/425 [00:00<?, ?it/s]

{'hit_rate': 0.8847058823529412, 'mrr': 0.7545098039215685}


  0%|          | 0/425 [00:00<?, ?it/s]

{'hit_rate': 0.9858823529411764, 'mrr': 0.9231372549019607}


In [ ]:
from itertools import product


question_boost_dict_values_to_test = [1]
answer_boost_dict_values_to_test = [1, 1.5, 2, 3, 5]
section_boost_dict_values_to_test = [0, 0.05, 0.1, 0.25, 0.5]

boost_dicts = [
    {
        "question": question_boost,
        "answer": answer_boost,
        "section": section_boost,
    }
    for question_boost, answer_boost, section_boost in product(
        question_boost_dict_values_to_test,
        answer_boost_dict_values_to_test,
        section_boost_dict_values_to_test,
    )
]

print(boost_dicts)

[{'question': 1, 'answer': 1, 'section': 0}, {'question': 1, 'answer': 1, 'section': 0.05}, {'question': 1, 'answer': 1, 'section': 0.1}, {'question': 1, 'answer': 1, 'section': 0.25}, {'question': 1, 'answer': 1, 'section': 0.5}, {'question': 1, 'answer': 1.5, 'section': 0}, {'question': 1, 'answer': 1.5, 'section': 0.05}, {'question': 1, 'answer': 1.5, 'section': 0.1}, {'question': 1, 'answer': 1.5, 'section': 0.25}, {'question': 1, 'answer': 1.5, 'section': 0.5}, {'question': 1, 'answer': 2, 'section': 0}, {'question': 1, 'answer': 2, 'section': 0.05}, {'question': 1, 'answer': 2, 'section': 0.1}, {'question': 1, 'answer': 2, 'section': 0.25}, {'question': 1, 'answer': 2, 'section': 0.5}, {'question': 1, 'answer': 3, 'section': 0}, {'question': 1, 'answer': 3, 'section': 0.05}, {'question': 1, 'answer': 3, 'section': 0.1}, {'question': 1, 'answer': 3, 'section': 0.25}, {'question': 1, 'answer': 3, 'section': 0.5}, {'question': 1, 'answer': 5, 'section': 0}, {'question': 1, 'answer':

In [37]:
from tqdm.auto import tqdm


class BoostDictEvaluationResults(TypedDict):
    boost_dict: dict[str, float]
    evaluation_results: dict[str, float]


evaluation_results_for_boost_dicts: list[BoostDictEvaluationResults] = []

for boost_dict in tqdm(boost_dicts):

    minsearch_lexical_search_tool = MinsearchLexicalSearchTool.from_documents(
        cast(list[JSONDocument], documents),
        text_fields=["question", "section", "answer"],
        keyword_fields=["course"],
        config=LexicalSearchConfig(
            num_results=5,
            boost_dict=boost_dict
        )
    )

    evaluation_results = evaluate(faq_ground_truth, minsearch_lexical_search_tool)

    evaluation_results_for_boost_dicts.append({
        "boost_dict": boost_dict,
        "evaluation_results": evaluation_results
    })

evaluation_results_for_boost_dicts = sorted(
    evaluation_results_for_boost_dicts,
    key=lambda evaluation_result: (
        evaluation_result["evaluation_results"]["hit_rate"],
        evaluation_result["evaluation_results"]["mrr"],
    ),
    reverse=True,
)

  0%|          | 0/25 [00:00<?, ?it/s]

In [38]:
for _ in evaluation_results_for_boost_dicts[:15]:
    print(_)

{'boost_dict': {'question': 1, 'answer': 3, 'section': 0}, 'evaluation_results': {'hit_rate': 0.9858823529411764, 'mrr': 0.8961960784313723}}
{'boost_dict': {'question': 1, 'answer': 3, 'section': 0.1}, 'evaluation_results': {'hit_rate': 0.9858823529411764, 'mrr': 0.8958431372549018}}
{'boost_dict': {'question': 1, 'answer': 3, 'section': 0.05}, 'evaluation_results': {'hit_rate': 0.9858823529411764, 'mrr': 0.8956078431372546}}
{'boost_dict': {'question': 1, 'answer': 3, 'section': 0.25}, 'evaluation_results': {'hit_rate': 0.9788235294117648, 'mrr': 0.89278431372549}}
{'boost_dict': {'question': 1, 'answer': 5, 'section': 0.1}, 'evaluation_results': {'hit_rate': 0.9788235294117648, 'mrr': 0.8859999999999998}}
{'boost_dict': {'question': 1, 'answer': 2, 'section': 0}, 'evaluation_results': {'hit_rate': 0.9764705882352941, 'mrr': 0.8889411764705879}}
{'boost_dict': {'question': 1, 'answer': 2, 'section': 0.05}, 'evaluation_results': {'hit_rate': 0.9764705882352941, 'mrr': 0.88874509803921